Imports and Setup

In [ ]:
import os
import json
import time
import pandas as pd
import google.generativeai as genai

API_KEY = "put your own key" 
genai.configure(api_key=API_KEY)
MODEL_NAME = "gemini-2.5-flash"
# Define your directories
BASE_DIR = os.getcwd()
PDF_DIRECTORY = os.path.join(BASE_DIR, "papers") #put only one articles in this folder to try the code since their is a limit on the number of requests
OUTPUT_EXCEL = "scanned_pdf_extraction.xlsx"

Define the Prompt

In [5]:
EXTRACTION_PROMPT = """
**Role:** You are an expert Material Scientist and Data Curator specializing in Organic Photovoltaics (OPV).

**Task:** Extract structured device data from the provided scientific text. You must identify every unique solar cell device configuration mentioned and extract its fabrication details and performance metrics into a JSON format.

**Critical Instruction for Device Separation:**
* Treat EVERY variation as a separate device entry.
* If a paper tests 3 different D:A ratios (e.g., 1:1, 1:1.2, 1:1.5), output **3 separate JSON objects**.
* If a paper tests 2 different annealing temperatures (e.g., 100°C vs 120°C), output **2 separate JSON objects**.
* Do not aggregate disparate conditions into one entry.

**Normalization Rules (Strictly Apply These):**
* **Solvents:** "CB" → "Chlorobenzene", "CF" → "Chloroform", "DCB/o-DCB" → "o-Dichlorobenzene", "DIO" → "1,8-Diiodooctane", "CN" → "Chloronaphthalene".
* **Units:** Convert all efficiencies to % (e.g., 0.15 → 15%). Convert thickness to nm.
* **Structure Inference:** If "Device Structure" is not explicitly stated, infer it from the layer order:
    * ITO/PEDOT:PSS/... → "Conventional"
    * ITO/ZnO/... or ITO/SnO2/... → "Inverted"

**Handling Metrics & Ranges:**
* If a range is given (e.g., "PCE: 15-16%"), extract the **maximum (champion)** value.
* If a value is missing, explicitly write "Not reported" (do not leave blank).

### TARGET JSON STRUCTURE
Return a list of JSON objects. strictly adhere to this schema:

```json
[
  {
    "device_id": "Device 1 (e.g., Optimal Ratio 1:1.2)",
    "architecture": {
      "type": "string (Conventional, Inverted, or Unknown)",
      "anode": "string",
      "hole_transport_layer": "string (e.g., PEDOT:PSS, MoO3)",
      "electron_transport_layer": "string (e.g., ZnO, PFN-Br)",
      "cathode": "string"
    },
    "active_layer": {
      "donor": "string (e.g., PM6)",
      "acceptor": "string (e.g., Y6, L8-BO)",
      "ratio_weight": "string (e.g., 1:1.2)",
      "thickness_nm": "float or 'Not reported'"
    },
    "processing": {
      "solvent": "string (Normalized, e.g., Chloroform)",
      "total_concentration_mg_ml": "float or 'Not reported'",
      "additive_name": "string (Normalized, e.g., 1,8-Diiodooctane)",
      "additive_volume_percent": "float or 'Not reported'",
      "deposition_method": "string (e.g., Spin-coating, Blade-coating)",
      "annealing_temperature_celsius": "float or 'Not reported'",
      "annealing_time_min": "float or 'Not reported'"
    },
    "metrics": {
      "pce_percent": "float (Champion value)",
      "voc_volts": "float",
      "jsc_ma_cm2": "float",
      "ff_percent": "float",
      "HUMO": "float",
      "LUMO": "float"
    },
    "notes": "string (Capture critical nuances, e.g., 'Device measured with mask', 'Ternary blend')"
  }
]
      
"""

Processing Functions

In [6]:
def process_pdf_natively(file_path):
    """
    Uploads the PDF directly to Gemini via the File API.
    This enables 'Native Multimodal' processing (reading text layer + layout).
    """
    model = genai.GenerativeModel(MODEL_NAME)
    file_name = os.path.basename(file_path)
    
    print(f"  - Uploading {file_name} to Gemini...")
    
    try:
        # 1. Upload the file
        uploaded_file = genai.upload_file(path=file_path, display_name=file_name)
        
        # 2. Wait for processing to complete
        while uploaded_file.state.name == "PROCESSING":
            print("    Processing file on server...", end="\r")
            time.sleep(2)
            uploaded_file = genai.get_file(uploaded_file.name)
            
        if uploaded_file.state.name == "FAILED":
            print(f"    ! File processing failed on server.")
            return []

        print(f"    File Ready. Sending prompt to model...")

        # 3. We send the file object AND the text prompt together
        response = model.generate_content(
            [uploaded_file, EXTRACTION_PROMPT],
            generation_config={"response_mime_type": "application/json"} # Enforce JSON output
        )
        
        # 4. Parse JSON result
        json_data = json.loads(response.text)
        
        # Ensure it's a list
        if isinstance(json_data, dict):
            return [json_data]
        return json_data

    except Exception as e:
        print(f"    ! Error processing {file_name}: {e}")
        return []

def flatten_json(json_data, filename):
    """Flattens nested JSON for Excel rows."""
    flat_rows = []
    for entry in json_data:
        if not entry: continue
        
        # Helper to safely get nested values
        proc = entry.get("processing", {})
        active = entry.get("active_layer", {})
        metrics = entry.get("metrics", {})
        arch = entry.get("architecture", {})

        row = {
            "Source File": filename,
            "Device ID": entry.get("device_id"),
            "Structure": arch.get("type"),
            "Donor": active.get("donor"),
            "Acceptor": active.get("acceptor"),
            "Ratio": active.get("ratio_weight"),
            "Solvent": proc.get("solvent"),
            "Additive": proc.get("additive_name"),
            "Annealing Temp": proc.get("annealing_temperature_celsius"),
            "PCE (%)": metrics.get("pce_percent"),
            "Voc (V)": metrics.get("voc_volts"),
            "Jsc (mA/cm2)": metrics.get("jsc_ma_cm2"),
            "FF (%)": metrics.get("ff_percent"),
            "Notes": entry.get("notes")
        }
        flat_rows.append(row)
    return flat_rows

Main Execution

In [7]:
files = [f for f in os.listdir(PDF_DIRECTORY) if f.endswith('.pdf')]
all_data = []

print(f"Found {len(files)} PDFs in '{PDF_DIRECTORY}'")

if files:
    for filename in files:
        file_path = os.path.join(PDF_DIRECTORY, filename)
        print(f"\nProcessing: {filename}")
        extracted_json = process_pdf_natively(file_path)
        if extracted_json:
            flat_data = flatten_json(extracted_json, filename)
            all_data.extend(flat_data)
            print(f"  -> Successfully extracted {len(flat_data)} devices.")
        else:
            print("  -> No data found or error occurred.")
            
    # Save to Excel
    if all_data:
        df = pd.DataFrame(all_data)
        df.to_excel(OUTPUT_EXCEL, index=False)
        print(f"\n✅ Success! Data saved to {OUTPUT_EXCEL}")
        print("First 5 rows preview:")
        display(df.head())
    else:
        print("\n No data extracted from any files.")
else:
    print("Please add PDF files to the 'papers' folder and run this cell again.")

Found 1 PDFs in 'c:\Users\mehdi\Downloads\solaire\papers'

Processing: information support_Article 6.pdf
  - Uploading information support_Article 6.pdf to Gemini...
    File Ready. Sending prompt to model...
  -> Successfully extracted 8 devices.

✅ Success! Data saved to scanned_pdf_extraction.xlsx
First 5 rows preview:


,Source File,Device ID,Structure,Donor,Acceptor,Ratio,Solvent,Additive,Annealing Temp,PCE (%),Voc (V),Jsc (mA/cm2),FF (%),Notes
0,information support_Article 6.pdf,PM6:BTP-eC9 (Binary) - 1:1.2,Conventional,PM6,BTP-eC9,1:1.2,"Chlorobenzene (for donor), Chloroform (for acc...","2,5-dibromo-3,4-difluorothiophene",100.0,17.55,0.843,26.78,77.74,Binary blend of PM6 and BTP-eC9. Acceptor BTP-...
1,information support_Article 6.pdf,PM6:BTP-eC9:Qx-Ac (Ternary) - 1:1:0.1,Conventional,PM6,"BTP-eC9, Qx-Ac",1:1:0.1,"Chlorobenzene (for donor), Chloroform (for acc...","2,5-dibromo-3,4-difluorothiophene",100.0,17.70,0.849,26.74,77.88,"Ternary blend of PM6, BTP-eC9, and Qx-Ac. Acce..."
2,information support_Article 6.pdf,PM6:BTP-eC9:Qx-Ac (Ternary) - 1:1:0.2,Conventional,PM6,"BTP-eC9, Qx-Ac",1:1:0.2,"Chlorobenzene (for donor), Chloroform (for acc...","2,5-dibromo-3,4-difluorothiophene",100.0,17.84,0.855,26.72,78.11,"Ternary blend of PM6, BTP-eC9, and Qx-Ac. Acce..."
3,information support_Article 6.pdf,PM6:BTP-eC9:Qx-Ac (Ternary - Optimal) - 1:0.8:0.4,Conventional,PM6,"BTP-eC9, Qx-Ac",1:0.8:0.4,"Chlorobenzene (for donor), Chloroform (for acc...","2,5-dibromo-3,4-difluorothiophene",100.0,18.51,0.874,26.69,79.35,"Ternary blend of PM6, BTP-eC9, and Qx-Ac at op..."
4,information support_Article 6.pdf,PM6:BTP-eC9:Qx-Ac (Ternary) - 1:0.6:0.6,Conventional,PM6,"BTP-eC9, Qx-Ac",1:0.6:0.6,"Chlorobenzene (for donor), Chloroform (for acc...","2,5-dibromo-3,4-difluorothiophene",100.0,17.53,0.887,26.22,75.36,"Ternary blend of PM6, BTP-eC9, and Qx-Ac. Acce..."
